# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description (ensure Dataset object methods are used)
print(f"Title : {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² Croissant package provides structured record sets (tables), where each record set has an `@id` and a list of fields/columns. We'll enumerate all available record sets and their key components by their `@id`.

In [ ]:
# Display all record sets and their fields by their @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name         : {rs.name}")
    print(f"  description  : {rs.description}")
    print("  Fields and their @id:")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name} ({field.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set and load into a DataFrame
# We'll list all @id's, then focus on the first record set for demonstration

dataframes = {}
recordset_ids = [rs.id for rs in dataset.record_sets]
print(f"Record set @ids found: {recordset_ids}\n")

# Set the primary record set for demonstration (here we choose the first one)
main_record_set_id = recordset_ids[0]
print(f"Loading data from main record set: {main_record_set_id}")

for record_set_id in recordset_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {record_set_id}")
    else:
        print(f"No data found for record set {record_set_id}")

# Display variables (fields/@ids) of main record set
if main_record_set_id in dataframes:
    print(f"\nColumns in DataFrame for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame found for main record set {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Operations include removing outliers, transforming data distributions, or grouping data by attributes for further analysis.

> **💡 To perform EDA, you can adapt numeric_field and group_field to available field `@id`s as seen above!**

In [ ]:
# Example: Filter by a numeric field, normalize, and group
# Identify a numeric field by @id (e.g. 'abundance', 'MW', 'coverage', etc.)
df = dataframes.get(main_record_set_id)

# For demonstration: pick a suitable numeric column, e.g., 'MW@id' or similar
# List numeric-looking columns
if df is not None:
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric columns available: {numeric_columns}")
    if numeric_columns:
        numeric_field = numeric_columns[0]   # choose the first numeric field as example
        threshold = df[numeric_field].mean()

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Choose a grouping field (categorical) if present
        group_candidates = df.select_dtypes(include=['object']).columns.tolist()
        print(f"\nCategorical (object) columns available: {group_candidates}")
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Visualize the distribution of the numeric field in the main record set
if df is not None and numeric_columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If a group field exists, visualize mean per group
    if group_field:
        plt.figure(figsize=(10,5))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        sns.barplot(x=group_means.index, y=group_means.values, palette='Set2')
        plt.xticks(rotation=90)
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrates loading, exploring, and analyzing the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. You can adapt the field names, filtering thresholds, and grouping attributes depending on your specific research or analysis goal.

For deeper analysis, consult the Croissant schema for custom field `@id`s, explore relationships between additional attributes, or connect to the original record sets for advanced workflow needs. Happy data science!